# Phase 2 — Kaggle GPU detection and tracking

Attach the RWF-2000, RLVS, and Hockey Fights Kaggle datasets as notebook inputs. Set the three input paths in Cell 3 to the dataset roots. Select a GPU accelerator and enable Internet for the first run so Ultralytics can fetch pretrained weights. Outputs are saved below `/kaggle/working/detections` and cache generation can resume after a session interruption.

## 1. Clone the project and install detection dependencies

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch

REPO_URL = 'https://github.com/Ayaan-Ali-Khan/Violence-Detection.git'
REPO_DIR = Path('/kaggle/working/Violence-Detection')
GIT_REF = 'main'  # Pin to a commit SHA for a report run.
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--tags'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GIT_REF], check=True)
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src')
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-kaggle.txt'], check=True)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU accelerator before running the full cache. CPU caching would be very slow.')
print('GPU:', torch.cuda.get_device_name(0))

## 2. Link the attached datasets

The paths below should point to the directories containing the class/split folders (or the flat Hockey Fights AVI files). Kaggle mounts them read-only; symlinks let the repository manifests keep their original `data/raw/...` paths.

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
print('Mounted Kaggle inputs:')
for p in sorted(INPUT_ROOT.iterdir()):
    print(' ', p)

# Edit these paths to match the inputs printed above.
DATASET_INPUTS = {
    'rwf2000': INPUT_ROOT / 'rwf2000',
    'rlvs': INPUT_ROOT / 'rlvs',
    'hockeyfights': INPUT_ROOT / 'hockeyfights',
}
RAW_DIR = REPO_DIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
for name, source in DATASET_INPUTS.items():
    if not source.exists():
        raise FileNotFoundError(f'Update DATASET_INPUTS[{name!r}]; not found: {source}')
    target = RAW_DIR / name
    if target.is_symlink():
        target.unlink()
    elif target.exists():
        raise FileExistsError(f'Remove or rename existing path before linking: {target}')
    target.symlink_to(source.resolve(), target_is_directory=True)
    print(f'{name}: {target} -> {source.resolve()}')

## 3. Benchmark model sizes and inspect a ByteTrack sample

In [ ]:
subprocess.run([sys.executable, 'scripts/benchmark_yolo.py'], check=True)
sample = sorted((RAW_DIR / 'rwf2000' / 'val' / 'Fight').glob('*.avi'))[0]
subprocess.run([sys.executable, 'scripts/smoke_track.py', '--video', str(sample), '--device', '0'], check=True)

## 4. Cache all RWF train/validation/test tracks

This includes every row in the RWF split manifest. The script writes per-clip compressed files, skips current caches, and resumes safely if rerun.

In [ ]:
DETECTION_DIR = Path('/kaggle/working/detections/yolo11n')
DETECTION_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, 'scripts/cache_tracks.py', 'data/splits/rwf2000.csv', '--cache-dir', str(DETECTION_DIR), '--stats-output', '/kaggle/working/detections/per_clip_stats.csv', '--device', '0'], check=True)

## 5. Cache RLVS and Hockey Fights evaluation tracks

In [ ]:
subprocess.run([sys.executable, 'scripts/cache_tracks.py', 'data/splits/rlvs_eval.csv', 'data/splits/hockeyfights_eval.csv', '--cache-dir', str(DETECTION_DIR), '--stats-output', '/kaggle/working/detections/per_clip_stats.csv', '--device', '0'], check=True)

## 6. Summarize detection statistics and inspect saved outputs

In [ ]:
subprocess.run([sys.executable, 'scripts/summarize_detection_stats.py', '--cache-dir', str(DETECTION_DIR), '--output', '/kaggle/working/detections/detection_stats.csv', '--report', '/kaggle/working/detections/detection_stats.md'], check=True)
for p in sorted(Path('/kaggle/working/detections').rglob('*')):
    if p.is_file():
        print(p, p.stat().st_size, 'bytes')